Instalação de dependências

In [ ]:
!pip uninstall -y tensorflow tensorflow-text tf-keras tensorflow-decision-forests ydf-tf keras keras-core keras-nlp
# Installing specific TensorFlow and tf-keras versions to ensure compatibility.
!pip install tensorflow tf-keras tensorflow-text --quiet
!pip install -q pandas scikit-learn matplotlib seaborn
!pip install -q requests tqdm
!pip uninstall -y albumentations albucore
!pip install albumentations==1.4.8 albucore==0.0.15

Found existing installation: albumentations 1.4.8
Uninstalling albumentations-1.4.8:
  Successfully uninstalled albumentations-1.4.8
Found existing installation: albucore 0.0.15
Uninstalling albucore-0.0.15:
  Successfully uninstalled albucore-0.0.15
  Using cached albumentations-1.4.8-py3-none-any.whl.metadata (37 kB)
  Using cached albucore-0.0.15-py3-none-any.whl.metadata (3.1 kB)
Using cached albumentations-1.4.8-py3-none-any.whl (156 kB)
Using cached albucore-0.0.15-py3-none-any.whl (9.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [albumentations]


Imports e acesso ao drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import albumentations as A
import tensorflow as tf
import cv2
import numpy as np
from matplotlib import pyplot as plt
import os
import pandas as pd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


- Oversampling no dataset de treino
- One-hot-enconding
- Criação de dataframes no tensorflow

In [ ]:
TRAIN_DIR = '/content/drive/MyDrive/TCC1/colab_data/isic/train'
TEST_DIR = '/content/drive/MyDrive/TCC1/colab_data/isic/test'
CSV_PATH = '/content/drive/MyDrive/TCC1/colab_data/isic/metadata.csv'

# 1. Carrega o CSV original
df = pd.read_csv(CSV_PATH)

# 2. Lê os arquivos físicos
files_train = set(os.listdir(TRAIN_DIR))
files_test = set(os.listdir(TEST_DIR))

# 3. Cria uma coluna temporária no DataFrame para comparação (.jpg)
df['nome_com_extensao'] = df['isic_id'] + '.jpg'

# 4. SEPARAÇÃO EM DOIS DATAFRAMES
df_train = df[df['nome_com_extensao'].isin(files_train)].copy()
df_test = df[df['nome_com_extensao'].isin(files_test)].copy()

# =====================================================================
# 5. MAPEAMENTO DE CLASSES (A coluna 'label_int' PRECISA ser criada aqui!)
# =====================================================================
mapa_classes = {'Benign': 0, 'Indeterminate': 1, 'Malignant': 2}
df_train['label_int'] = df_train['diagnosis_1'].map(mapa_classes)
df_test['label_int'] = df_test['diagnosis_1'].map(mapa_classes)

# =====================================================================
# BALANCEAMENTO VIA OVERSAMPLING (DADOS EQUILIBRADOS ANTES DO TREINO)
# =====================================================================
print(f"Antes do Balanceamento - Treino: {df_train['diagnosis_1'].value_counts().to_dict()}")

# Encontra a quantidade da classe majoritária (Benign = 8471)
maior_classe_qtd = df_train['label_int'].value_counts().max()

# Replica as classes minoritárias para que fiquem com o mesmo tamanho da maior
df_balanced_list = []
for classe_id in range(3):
    df_da_classe = df_train[df_train['label_int'] == classe_id]

    # Se a classe for menor que a maioria, fazemos amostragem com reposição (duplica as fotos)
    df_classe_replicada = df_da_classe.sample(maior_classe_qtd, replace=True, random_state=42)
    df_balanced_list.append(df_classe_replicada)

# Junta tudo de volta e embaralha as linhas
df_train = pd.concat(df_balanced_list, axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Após o Balanceamento - Treino: {df_train['diagnosis_1'].value_counts().to_dict()}")
# =====================================================================

print(f"Imagens finais para Treino (Após Oversampling): {len(df_train)}")
print(f"Imagens encontradas para Teste: {len(df_test)}")

# 6. ONE-HOT ENCODING
labels_train_one_hot = tf.keras.utils.to_categorical(df_train['label_int'].values, num_classes=3)
labels_test_one_hot = tf.keras.utils.to_categorical(df_test['label_int'].values, num_classes=3)

# 7. CRIAÇÃO DOS CAMINHOS
file_path_train = [os.path.join(TRAIN_DIR, fname) for fname in df_train['nome_com_extensao']]
file_path_test = [os.path.join(TEST_DIR, fname) for fname in df_test['nome_com_extensao']]

# 8. CONSTRUÇÃO DOS DATASETS DO TENSORFLOW
train_dataset = tf.data.Dataset.from_tensor_slices((file_path_train, labels_train_one_hot))
test_dataset = tf.data.Dataset.from_tensor_slices((file_path_test, labels_test_one_hot))



Antes do Balanceamento - Treino: {'Benign': 8471, 'Malignant': 200, 'Indeterminate': 42}
Após o Balanceamento - Treino: {'Malignant': 8471, 'Benign': 8471, 'Indeterminate': 8471}
Imagens finais para Treino (Após Oversampling): 25413
Imagens encontradas para Teste: 2269


Redimensionamento de imagens e aplicação do CoarseDropout

In [ ]:
# Definição das constantes
IMG_SIZE = 380  # Nativo do EfficientNetB4

train_transform = A.Compose([
    # 1. Redimensionamento para 380x380
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),

    # 2. Rotações, Espelhamentos e Cisalhamento (Shear)
    A.HorizontalFlip(p=0.5), # 50% de chance
    A.VerticalFlip(p=0.5),   # 50% de chance
    A.Rotate(limit=30, p=0.7), # Rotaciona até 30 graus
    # O cisalhamento (shear) no Albumentations é feito via Affine
    A.Affine(shear={'x': (-20, 20), 'y': (-20, 20)}, p=0.5, fit_output=False),

    # 3. CoarseDropout (Simula oclusões/pêlos/bolhas)
    # Cria até 8 buracos quadrados pretos na imagem
    A.CoarseDropout(
        max_holes=8,
        max_height=32,
        max_width=32,
        min_holes=2,
        min_height=8,
        min_width=8,
        fill_value=0, # Cor do buraco (0 = preto)
        p=0.5
    ),

    # 4. Normalização e Ajuste de Escala
    # Normaliza usando a média e desvio padrão do ImageNet
    # Converte os pixels de [0, 255] para float32
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
        max_pixel_value=255.0,
        p=1.0
    ),
])

Aplicação das transformações e geração dos datasets finais no tensorflow

In [ ]:
def aug_fn(image):
    """
    Função auxiliar que aplica as transformações de forma segura.
    Garante que o TensorFlow passe a imagem como um array NumPy puro.
    """
    # 1. Garante que a imagem está no formato NumPy e do tipo uint8 (esperado pelo Albumentations)
    img_np = np.array(image, dtype=np.uint8)

    # 2. Passa a imagem explicitamente usando o argumento 'image='
    aug_data = train_transform(image=img_np)

    # 3. Recupera a imagem transformada e converte para float32 (esperado pela EfficientNet)
    aug_img = aug_data["image"]
    return aug_img.astype(np.float32)

def process_data(file_path, label):
    # 1. Ler o arquivo de imagem
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)

    # 2. Aplicar o Albumentations via numpy_function
    # O TensorFlow chama a função python 'aug_fn'
    aug_img = tf.numpy_function(func=aug_fn, inp=[img], Tout=tf.float32)

    # 3. Recuperar a forma (Shape) do Tensor
    # O tf.numpy_function perde a informação de shape, precisamos redefinir
    aug_img.set_shape((IMG_SIZE, IMG_SIZE, 3))

    return aug_img, label

# 5. Criação do Dataset final do TensorFlow
train_dataset = tf.data.Dataset.from_tensor_slices((file_path_train, labels_train_one_hot))
test_dataset = tf.data.Dataset.from_tensor_slices((file_path_test, labels_test_one_hot))
train_dataset = train_dataset.map(process_data, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.map(process_data, num_parallel_calls=tf.data.AUTOTUNE)# Pipeline com Albumentations
train_dataset = train_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(32).prefetch(tf.data.AUTOTUNE)


Primeira tentativa de mitigar o desbalancemento de dados. Inutilizada após implementação do oversampling.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

print("Calculando pesos das classes para balanceamento...")

# Pega os rótulos reais de treino (já que eles estão em formato inteiro 0, 1, 2)
y_train_inteiros = df_train['label_int'].values
classes_unicas = np.unique(y_train_inteiros)

# Calcula os pesos balanceados
pesos = compute_class_weight(
    class_weight='balanced',
    classes=classes_unicas,
    y=y_train_inteiros
)

# Converte para o formato de dicionário que o Keras exige
pesos_das_classes = {i: peso for i, peso in zip(classes_unicas, pesos)}

print("Pesos calculados:")
for classe, peso in pesos_das_classes.items():
    nome_classe = list(mapa_classes.keys())[list(mapa_classes.values()).index(classe)]
    print(f"Classe {nome_classe}: {peso:.2f}")

Calculando pesos das classes para balanceamento...
Pesos calculados:
Classe Benign: 0.34
Classe Indeterminate: 69.15
Classe Malignant: 14.52


Checkpoint de salvamento do modelo já treinado.

In [ ]:
import os

# Diretório para salvar os resultados do TCC
SALVAMENTO_DIR = '/content/drive/MyDrive/TCC1/colab_data/isic/results'
os.makedirs(SALVAMENTO_DIR, exist_ok=True)

callbacks_list = [
    # 1. Salva automaticamente o melhor modelo no Drive ao longo das épocas
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(SALVAMENTO_DIR, 'melhor_modelo_efficientnetb4.h5'),
        monitor='val_loss',         # Monitora a perda na validação
        save_best_only=True,        # Só substitui o arquivo se o modelo melhorar
        mode='min',
        verbose=1
    ),

    # 2. Interrompe o treinamento se o modelo parar de evoluir (Evita Overfitting)
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=12,                 # Se por 7 épocas a val_loss não cair, ele para
        restore_best_weights=True,  # Ao parar, recupera os pesos da melhor época
        verbose=1
    ),

    # 3. Reduz o Learning Rate se o treinamento estagnar em um patamar
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,                 # Multiplica o LR atual por 0.2 (reduz 5x)
        patience=5,                 # Espera 3 épocas de estagnação antes de reduzir
        min_lr=1e-6,
        verbose=1
    )
]

Carregamento do último modelo salvo para reiniciar o treinamento após uma interrupção.

In [ ]:
CAMINHO_MODELO_DRIVE = '/content/drive/MyDrive/TCC1/colab_data/isic/results/melhor_modelo_efficientnetb4.h5'

# Carrega o modelo (o Keras vai trazer os pesos salvos até o momento do crash)
base_model = tf.keras.applications.EfficientNetB4(weights='imagenet', include_top=False, input_shape=(380, 380, 3))
base_model.trainable = False
model = tf.keras.models.load_model(CAMINHO_MODELO_DRIVE)
print("✅ Modelo carregado com sucesso!")


71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


✅ Modelo carregado com sucesso!


Aumento de camadas disponíveis para treinamento.

In [ ]:
import tensorflow as tf

CAMINHO_MODELO_DRIVE = '/content/drive/MyDrive/TCC1/colab_data/isic/results/melhor_modelo_efficientnetb4.h5'

print("Resgatando o modelo do Drive para Fase 3 (Deep Fine-Tuning)...")
model = tf.keras.models.load_model(CAMINHO_MODELO_DRIVE)

model.trainable = True

# AUMENTAMOS DE 53 PARA 150 CAMADAS (Permite que a rede ajuste texturas médias e finas)
camadas_ativas = 150

for layer in model.layers[:-camadas_ativas]:
    layer.trainable = False

for layer in model.layers[-camadas_ativas:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0), # Mantém a Focal Loss
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)

print(f"Total de pesos/variáveis agora treináveis: {len(model.trainable_variables)}")

history_fase3 = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=70,
    initial_epoch=60,   # Continua de onde parou
    callbacks=callbacks_list
)

Resgatando o modelo do Drive para Fase 3 (Deep Fine-Tuning)...


Total de pesos/variáveis agora treináveis: 132
Epoch 61/70
795/795 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7898 - loss: 0.0437 - recall: 0.7053
Epoch 61: val_loss improved from None to 0.05543, saving model to /content/drive/MyDrive/TCC1/colab_data/isic/results/melhor_modelo_efficientnetb4.h5



Epoch 61: finished saving model to /content/drive/MyDrive/TCC1/colab_data/isic/results/melhor_modelo_efficientnetb4.h5
795/795 ━━━━━━━━━━━━━━━━━━━━ 4834s 6s/step - accuracy: 0.7918 - loss: 0.0434 - recall: 0.7054 - val_accuracy: 0.7069 - val_loss: 0.0554 - val_recall: 0.6457 - learning_rate: 5.0000e-06
Epoch 62/70
795/795 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7902 - loss: 0.0432 - recall: 0.7081
Epoch 62: val_loss improved from 0.05543 to 0.05518, saving model to /content/drive/MyDrive/TCC1/colab_data/isic/results/melhor_modelo_efficientnetb4.h5



Epoch 62: finished saving model to /content/drive/MyDrive/TCC1/colab_data/isic/results/melhor_modelo_efficientnetb4.h5
795/795 ━━━━━━━━━━━━━━━━━━━━ 1526s 2s/step - accuracy: 0.7914 - loss: 0.0432 - recall: 0.7084 - val_accuracy: 0.7060 - val_loss: 0.0552 - val_recall: 0.6448 - learning_rate: 5.0000e-06
Epoch 63/70
795/795 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7953 - loss: 0.0426 - recall: 0.7148
Epoch 63: val_loss improved from 0.05518 to 0.04913, saving model to /content/drive/MyDrive/TCC1/colab_data/isic/results/melhor_modelo_efficientnetb4.h5



Epoch 63: finished saving model to /content/drive/MyDrive/TCC1/colab_data/isic/results/melhor_modelo_efficientnetb4.h5
795/795 ━━━━━━━━━━━━━━━━━━━━ 1518s 2s/step - accuracy: 0.7951 - loss: 0.0427 - recall: 0.7150 - val_accuracy: 0.7422 - val_loss: 0.0491 - val_recall: 0.6959 - learning_rate: 5.0000e-06
Epoch 64/70
